<a href="https://colab.research.google.com/github/peterhgruber/python-intro-colab/blob/main/Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Embeddings

Peter H. Gruber • https://www.abitof.ai

Embeddings map words or sentences to  vectors in the semantic space. So words become high-dimensional vectors of real numbers. By placing text in a vector space, we relate the (euclidian) distance to the concept of similar meaning.

## Setup
This may take a moment and needs to be done every time you run the notebook.

In [ ]:
# 1. Install necessary libraries (no API keys required)
!pip install -q gensim sentence-transformers scikit-learn matplotlib seaborn

## 1. Word Embeddings and Vector Arithmetic

We will use a small pre-trained GloVe model.

In [ ]:
import gensim.downloader as api

# Load a small 50-dimensional word embedding model (~66MB)
model = api.load("glove-wiki-gigaword-50")
print("Model loaded successfully!")

In [ ]:
# The classic embedding trick: King - Man + Woman = ?
result = model.most_similar(positive=['king', 'woman'], negative=['man'], topn=1)

print(f"king - man + woman = {result[0][0]} (similarity: {result[0][1]:.2f})")

### Visualizing a Vector
What does an embedding actually look like inside the computer?

In [ ]:
import matplotlib.pyplot as plt

# Extract the 50-dimensional vectors for 'king' and 'queen'
king_vector = model['king']
queen_vector = model['queen']
diff_vector = king_vector - queen_vector

# Create a faceted plot with 3 rows
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, sharey=True)

# Plot 'king' on the first row
axes[0].bar(range(len(king_vector)), king_vector, color='teal')
axes[0].set_title("The 50 dimensions of the word 'king'")
axes[0].set_ylabel("Value")

# Plot 'queen' on the second row
axes[1].bar(range(len(queen_vector)), queen_vector, color='purple')
axes[1].set_title("The 50 dimensions of the word 'queen'")
axes[1].set_ylabel("Value")

plt.tight_layout()
plt.show()


## 2. Sentence Embeddings

Instead of single words, we often want to embed entire sentences to capture context. We use a local `SentenceTransformer` model.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a small, fast local model for sentence embeddings
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')

# 5 sample sentences (4 about weather, 1 completely different)
sentences = [
    "The sun is shining brightly today.",
    "It is very hot and sunny outside.",
    "Heavy rain is expected this afternoon.",
    "A dark thunderstorm is approaching the city.",
    "I love eating pepperoni pizza on Fridays."
]

# Generate the embeddings (each will be a 384-dimensional vector)
embeddings = sentence_model.encode(sentences)
print(f"Shape of embeddings array: {embeddings.shape}")

### Sentence Similarity (Distance)
Let's calculate the cosine similarity between our 5 sentences. This is literally

$$d_{x,y} = \frac{1}{N} \sum_{i=1}^{N} (x_i - y_i) ^2$$

With $N=384$ in our case.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# Calculate Euclidean distance "by hand"
n_sentences = len(sentences)
euclidean_matrix = np.zeros((n_sentences, n_sentences))

for i in range(n_sentences):
    for j in range(n_sentences):
        # distance = sqrt(sum((vec_i - vec_j)^2))
        diff = embeddings[i] - embeddings[j]
        euclidean_matrix[i, j] = np.sqrt(np.sum(diff**2))

print("Euclidean Distances ('by hand'):")
print(np.round(euclidean_matrix, 2))
print("\n--------------------------------------------------\n")

# Plot a heatmap of the Euclidean distances
plt.figure(figsize=(8, 5))
sns.heatmap(euclidean_matrix, annot=True, xticklabels=sentences, yticklabels=sentences, cmap="Reds", cbar=True)
plt.title("Euclidean Distance Between Sentences")
plt.xticks(rotation=45, ha='right')
plt.show()


In [ ]:
similarity_matrix = cosine_similarity(embeddings)

# Plot a heatmap of the similarities
plt.figure(figsize=(8, 5))
sns.heatmap(similarity_matrix, annot=True, xticklabels=sentences, yticklabels=sentences, cmap="YlGnBu", cbar=True)
plt.title("Cosine Similarity Between Sentences")
plt.xticks(rotation=45, ha='right')
plt.show()

### PCA Visualization
We can compress the 384 dimensions down to 3 dimensions using Principal Component Analysis (PCA) to visualize them in 3D space.

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Reduce 384 dimensions back to 3 dimensions
pca = PCA(n_components=3)
pca_result = pca.fit_transform(embeddings)

# Plotting: 5 lines for 5 sentences across PC1, PC2, PC3
fig, ax = plt.subplots(figsize=(10, 6))
pc_labels = ['PC1', 'PC2', 'PC3']

for i, sentence in enumerate(sentences):
    short_label = sentence[:30] + "..."
    ax.plot(pc_labels, pca_result[i], marker='o', linewidth=2, label=short_label)

ax.set_title("Principal Component Values for Each Sentence")
ax.set_xlabel("Principal Component")
ax.set_ylabel("PC Value")

# Place the legend outside the plot for better readability
ax.legend(title="Sentences", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 2D Scatter Plot
Visualizing the first two principal components as a scatter plot helps us see how the sentences cluster together based on their semantic meaning.

In [ ]:
# Plotting PC1 vs PC2 as a 2D scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pca_result[:, 0], pca_result[:, 1], s=100, color='royalblue')

# Add short labels to the points
for i, sentence in enumerate(sentences):
    short_label = sentence[:25] + "..."
    ax.text(pca_result[i, 0] + 0.02, pca_result[i, 1] + 0.02, short_label, size=9)

ax.set_title("Sentence Embeddings: PC1 vs PC2")
ax.set_xlabel("Principal Component 1")
ax.set_ylabel("Principal Component 2")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()